# Chapter 13 — Decoding and Sampling

Every number in Chapter 13, recomputed, and every sampler checked through `tfs.model.sample` (the
course's implementation, temperature → top-k → top-p → min-p). Each cell **asserts** what it checks,
so `python tests/run_notebooks.py 13` fails if the chapter and the code disagree. Exercises 13.2 and
13.3 use a tiny character model trained here for a few seconds on the bundled offline corpus instead
of the Chapter 11 Shakespeare model or GPT-2: a small analogue with the same shape of result.

In [ ]:
import math, sys, os
sys.path.insert(0, os.path.abspath(".."))      # run from notebooks/
sys.path.insert(0, os.path.abspath("."))       # or from the repo root
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(0)
torch.set_num_threads(min(4, torch.get_num_threads()))
from tfs.model import Config, Model, sample
logits = torch.tensor([3., 2., 1., 0., -1.])          # mat floor rug sky idea

## The worked example — one distribution, four samplers

In [ ]:
r3 = lambda t: [round(v, 3) for v in t.tolist()]
p = logits.softmax(-1)
temps = {0.5: [0.865, 0.117, 0.016, 0.002, 0.0], 1.0: [0.636, 0.234, 0.086, 0.032, 0.012],
         2.0: [0.429, 0.26, 0.158, 0.096, 0.058]}
for T, want in temps.items():
    print(f"T={T}", r3((logits / T).softmax(-1)))
    assert r3((logits / T).softmax(-1)) == want
assert abs(p[0] / p[1] - math.e) < 1e-5                          # one logit unit = e x probability
p2 = (logits / 2).softmax(-1)
assert round(1 / p2[4].item()) == 17                               # "idea": 1 in 17 at T = 2
assert r3(p[:2] / p[:2].sum()) == [0.731, 0.269]                  # top-k = 2, renormalised
assert r3(p.cumsum(0)) == [0.636, 0.871, 0.957, 0.988, 1.0]
kept = int(((p.cumsum(0) - p) < 0.9).sum())
assert kept == 3                                                    # top-p 0.9 keeps mat, floor, rug
sure = torch.tensor([0.98, 0.01, 0.005, 0.005])
assert int(((sure.cumsum(0) - sure) < 0.9).sum()) == 1              # a confident model keeps 1

## Exercise 13.1 — `tfs.model.sample`: T = 0 is argmax, and each filter shrinks the support

20,000 draws per setting; the empirical support must be exactly what the worked example predicts,
and the frequencies must match the renormalised probabilities.

In [ ]:
torch.manual_seed(0)
L = logits.expand(20_000, 5)
def draws(**kw):
    return sample(L, **kw).flatten()
assert (draws(temperature=0) == 0).all()
assert (draws(temperature=0, top_p=0.9) == 0).all()                 # quiz: T=0 decides everything
assert set(draws(top_k=2).tolist()) == {0, 1}
assert set(draws(top_p=0.9).tolist()) == {0, 1, 2}
assert set(draws(min_p=0.1).tolist()) == {0, 1, 2}                  # keep p >= 0.1 * 0.636
assert set(draws().tolist()) == {0, 1, 2, 3, 4}
freq = torch.bincount(draws(top_k=2), minlength=5).float() / 20_000
print("top-k=2 frequencies", r3(freq))
assert abs(freq[0] - 0.731) < 0.015 and abs(freq[1] - 0.269) < 0.015
freq = torch.bincount(draws(temperature=2.0), minlength=5).float() / 20_000
assert torch.allclose(freq, p2, atol=0.015)
# order matters: temperature before top-p. At T=2 the 0.9 nucleus is 4 tokens, not 3
assert set(draws(temperature=2.0, top_p=0.9).tolist()) == {0, 1, 2, 3}

## Temperature divides logits, not probabilities

Dividing the probabilities by T and renormalising is not a softer distribution — it is the same one.

In [ ]:
wrong = (p / 2.0) / (p / 2.0).sum()
assert torch.allclose(wrong, p) and not torch.allclose(p2, p)

## Repetition penalty (applied to logits) and constrained decoding (mask to −inf)

In [ ]:
lg = torch.tensor([2.0, -1.0, 0.5, 0.1])
pen = lg.clone()
for tok in {0, 1}:                                   # already generated
    pen[tok] /= 1.1 if pen[tok] > 0 else 1 / 1.1
print(lg.tolist(), "->", [round(v, 4) for v in pen.tolist()])
assert pen[0] < lg[0] and pen[1] < lg[1]             # both logits pushed down, positive or negative
assert pen.softmax(-1)[0] < lg.softmax(-1)[0]        # (a probability is relative: the others rise)
allowed = torch.tensor([True, False, True, False])
cons = lg.masked_fill(~allowed, float("-inf"))
torch.manual_seed(0)
assert set(sample(cons.expand(5000, 4)).flatten().tolist()) <= {0, 2}   # invalid tokens: p = 0

## 13.3 — beam search finds a likelier sequence than greedy (and costs k×)

A toy two-step model where the greedy first choice leads to a flat second step. Beam search with
k = 2 finds the higher-probability sequence.

In [ ]:
step1 = torch.tensor([0.6, 0.4])                          # P(first token)
step2 = {0: torch.tensor([0.35, 0.35, 0.3]), 1: torch.tensor([0.9, 0.05, 0.05])}
greedy_first = int(step1.argmax())
greedy = (greedy_first, int(step2[greedy_first].argmax()))
seq_p = lambda s: (step1[s[0]] * step2[s[0]][s[1]]).item()
beams = sorted(((a, b) for a in (0, 1) for b in range(3)), key=seq_p, reverse=True)[:2]
print(f"greedy {greedy} p={seq_p(greedy):.3f}   beam best {beams[0]} p={seq_p(beams[0]):.3f}")
assert seq_p(beams[0]) > seq_p(greedy)

## Exercises 13.2 and 13.3 — temperature, diversity and surprisal on a tiny trained model

A character model trained for a few seconds on the offline corpus (an analogue of the Chapter 11
model). At each temperature: 250 generated characters, their distinct-trigram ratio, and the longest
repeated substring. Then the average −log p the model assigns to greedy text, nucleus text and
held-out real text (Holtzman et al.'s comparison).

On this tiny model the ordering matches the chapter (greedy ≪ nucleus < real text, and nucleus is the
closer of the two) but nucleus text is not *close* to the real-text value as it is for GPT-2: a
few-second model is over-confident on its own samples. The asserts check the ordering only.

In [ ]:
from tfs.train import _fallback_corpus, get_batch
from tfs.tokenizer import CharTokenizer
text = _fallback_corpus()
ctok = CharTokenizer(text)
data = torch.tensor(ctok.encode(text))
train, held = data[:300_000], data[300_000:]
torch.manual_seed(0)
lm = Model(Config(vocab_size=ctok.vocab_size, d_model=64, n_layer=2, n_head=4, n_kv_head=2, d_head=16,
                  d_ff=192, max_T=512))
opt = torch.optim.AdamW(lm.parameters(), lr=3e-3)
for step in range(200):
    x = get_batch(train, 16, 96)
    _, loss, _ = lm(x, targets=x)
    opt.zero_grad(); loss.backward(); opt.step()
print(f"tiny LM trained: loss {loss.item():.3f}")
def longest_repeat(s):
    best = 0
    for n in range(1, len(s)):
        seen = {s[i:i + n] for i in range(len(s) - n + 1)}
        if len(seen) == len(s) - n + 1:
            break
        best = n
    return best
prompt = held[:16][None]
stats = {}
for T in (0.1, 0.5, 0.8, 1.0, 1.3, 2.0):
    torch.manual_seed(1)
    out = ctok.decode(lm.generate(prompt.clone(), 250, temperature=T)[0, 16:].tolist())
    tri = [out[i:i + 3] for i in range(len(out) - 2)]
    stats[T] = (len(set(tri)) / len(tri), longest_repeat(out))
    print(f"T={T:3}: distinct-trigram ratio {stats[T][0]:.2f}  longest repeat {stats[T][1]:3d}   {out[:50]!r}")
assert stats[0.1][0] < stats[1.0][0] < stats[2.0][0]                 # diversity rises with T
assert stats[0.1][1] > 3 * stats[2.0][1]                              # low T loops

In [ ]:
@torch.no_grad()
def surprisal(ids):                      # mean -log p per token, under the model
    lg, _, _ = lm(ids[None])
    return F.cross_entropy(lg[0, :-1], ids[1:]).item()
n = 300
torch.manual_seed(2)
greedy = lm.generate(prompt.clone(), n, temperature=0)[0]
nucleus = lm.generate(prompt.clone(), n, temperature=1.0, top_p=0.95)[0]
human = sum(surprisal(held[k * 500:(k + 1) * 500]) for k in range(4)) / 4     # 2000 real characters
s = {"greedy": surprisal(greedy), "top-p 0.95": surprisal(nucleus), "held-out text": human}
print({k: round(v, 3) for k, v in s.items()})
assert s["greedy"] < 0.5 * s["held-out text"]                        # far too likely to be "human"
assert abs(s["top-p 0.95"] - s["held-out text"]) < abs(s["greedy"] - s["held-out text"])